In [1]:
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Load data
df = pd.read_csv('train.csv')
target_labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Update missing rows
df['comment_text'] = df['comment_text'].fillna("missing comments")

In [3]:
df['cleaned'] = df['comment_text'].astype(str).str.lower().str.replace('[^a-zA-Z0-9 ]', '', regex=True)

In [4]:
df = df[df['cleaned'].str.strip() != '']

In [5]:
# Build Tokenizer Vocabulary (Top 10,000 words)
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<UNK>")
tokenizer.fit_on_texts(df['cleaned'])

In [6]:
# Convert sentences to numeric sequences and pad them
sequences = tokenizer.texts_to_sequences(df['cleaned'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
Y = df[target_labels].values

In [7]:
# Save the tokenizer asset immediately for Streamlit
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print(f"Shape of input data matrix: {X.shape}")

Shape of input data matrix: (159571, 100)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam

In [10]:
# Build the model structure layout
model = Sequential([
    #Embedding(input_dim=max_words + 2, output_dim=64, input_length=max_len),
    Embedding(input_dim=max_words + 2, output_dim=64, input_length=max_len),
    LSTM(128, return_sequences=False),
    Dense(64, activation='relu'),
    Dense(6, activation='sigmoid') # Sigmoid calculates independent binary targets
])

In [11]:
# Compile model tracking metrics
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [12]:
# Train the model for 2 clean cycles
print("\nStarting Training Processing...")
model.fit(X, Y, batch_size=128, epochs=2, validation_split=0.1)

# Save the finished model weights
model.save('lstm_toxic_model.h5')
print("\nModel saved successfully as 'lstm_toxic_model.h5'!")


Starting Training Processing...
Epoch 1/2
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 219s 190ms/step - accuracy: 0.9916 - loss: 0.1071 - val_accuracy: 0.9940 - val_loss: 0.0556
Epoch 2/2
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 191s 170ms/step - accuracy: 0.9942 - loss: 0.0521 - val_accuracy: 0.9940 - val_loss: 0.0527



Model saved successfully as 'lstm_toxic_model.h5'!


In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam

In [15]:
model_birnn = Sequential([
    #Embedding(input_dim=10002, output_dim=64, input_length=100),
    Embedding(input_dim=10002, output_dim=64),
    # Wrapping SimpleRNN inside Bidirectional double-checks context from both ends
    Bidirectional(SimpleRNN(64, return_sequences=False)), 
    Dense(64, activation='relu'),
    Dense(6, activation='sigmoid')
])

In [16]:
model_birnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model_birnn.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [17]:
#Bidirectional RNN Training
model_birnn.fit(X, Y, batch_size=128, epochs=2, validation_split=0.1)

# Save it to a unique file
model_birnn.save('birnn_toxic_model.h5')

Epoch 1/2
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 72s 59ms/step - accuracy: 0.9846 - loss: 0.0844 - val_accuracy: 0.9940 - val_loss: 0.0606
Epoch 2/2
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 66s 58ms/step - accuracy: 0.9927 - loss: 0.0535 - val_accuracy: 0.9940 - val_loss: 0.0695



Model saved successfully as 'birnn_toxic_model.h5'!
